In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from pathlib import PureWindowsPath

# ── device and config ─────────────────────────────────────────────────
device = (torch.device("mps") if torch.backends.mps.is_available() else
          torch.device("cuda") if torch.cuda.is_available() else
          torch.device("cpu"))
print("Device:", device)

PATCH_SIZE   = 224
BATCH_SIZE   = 16
MAX_EPOCHS   = 30
LR           = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS  = 0
THRESHOLD    = 0.5

splits_root  = Path("/kaggle/input/datasets/rohinipa1cb/splits-for-her2cancer/splits_stratified")
patches_root = Path("/kaggle/input/datasets/rohinipa1cb/patches-for-her2-cancer/patches")
out_root     = Path("her2_results"); out_root.mkdir(exist_ok=True)

# ── patch index ───────────────────────────────────────────────────────


patch_index = pd.read_csv(patches_root / "patch_index.csv")
patch_index["spot_id"] = patch_index["spot_id"].astype(str)
patch_index["patient"] = patch_index["patient"].astype(int)

# rebuild each memmap path from its file name, under the Kaggle patches folder
patch_index["mmap_path"] = patch_index["mmap_path"].apply(
    lambda p: str(patches_root / PureWindowsPath(p).name))

FILE_COUNTS = patch_index.groupby("mmap_path").size().to_dict()

print(len(FILE_COUNTS), "memmap files indexed")
print(next(iter(FILE_COUNTS)))

# ── loading spot tables ───────────────────────────────────────────────
def load_spots(path, her2_map):
    df = pd.read_csv(path, index_col=0)
    df.index.name = "spot_id"
    df = df.reset_index()
    df["spot_id"] = df["spot_id"].astype(str)
    df["patient"] = df["patient"].astype(int)
    df["label"]   = df["patient"].map(her2_map).map({"HER2-": 0, "HER2+": 1})
    assert df["label"].notna().all(), f"{path}: patient without HER2 label"
    return df
def spot_file(folder, name):
    for fn in (f"{name}.csv.gz", f"{name}.csv"):
        if (folder / fn).exists():
            return folder / fn
    raise FileNotFoundError(f"{name} not found in {folder}")
# ── dataset ───────────────────────────────────────────────────────────
class HER2Dataset(Dataset):
    def __init__(self, spots, augment=False, mean=None, std=None):
        m = spots.merge(patch_index[["spot_id", "patient", "section",
                                     "mmap_path", "mmap_idx"]],
                        on=["spot_id", "patient", "section"], how="inner")
        assert len(m) == len(spots), f"{len(spots) - len(m)} spots have no patch"

        self.meta   = m[["spot_id", "patient", "section", "label"]].reset_index(drop=True)
        self.paths  = m["mmap_path"].values
        self.idx    = m["mmap_idx"].astype(int).values     # row inside each section's memmap
        self.labels = m["label"].values.astype(np.float32)
        self.mmaps  = {p: np.memmap(p, dtype="uint8", mode="r",
                                    shape=(FILE_COUNTS[p], PATCH_SIZE, PATCH_SIZE, 3))
                       for p in np.unique(self.paths)}
        self.augment = augment
        self.mean = None if mean is None else torch.tensor(mean).view(3, 1, 1)
        self.std  = None if std  is None else torch.tensor(std).view(3, 1, 1)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        patch = np.array(self.mmaps[self.paths[i]][self.idx[i]])   # writable copy
        x = torch.from_numpy(patch).permute(2, 0, 1).float() / 255.0
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            x = torch.rot90(x, torch.randint(0, 4, (1,)).item(), dims=[1, 2])
            for c in range(3):
                x[c] = (x[c] * (0.8 + 0.4 * torch.rand(1).item())).clamp(0, 1)
        if self.mean is not None:
            x = (x - self.mean) / self.std
        return x, torch.tensor(self.labels[i])

def channel_stats(ds):
    """Per-channel mean and std over training patches (no augmentation)."""
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    s  = torch.zeros(3, dtype=torch.float64)
    ss = torch.zeros(3, dtype=torch.float64)
    n  = 0
    for x, _ in dl:
        x = x.double()
        s  += x.sum(dim=(0, 2, 3))
        ss += (x ** 2).sum(dim=(0, 2, 3))
        n  += x.shape[0] * x.shape[2] * x.shape[3]
    mean = s / n
    std  = torch.sqrt(ss / n - mean ** 2).clamp(min=1e-6)
    return mean.float().tolist(), std.float().tolist()

# ── model ─────────────────────────────────────────────────────────────
def conv_block(i, o):
    return nn.Sequential(
        nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
        nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(),
        nn.MaxPool2d(2))

class HER2CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(conv_block(3, 32), conv_block(32, 64),
                                     conv_block(64, 128), conv_block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256, 1))

    def forward(self, x):
        return self.head(self.encoder(x))   # one logit per spot

# ── prediction helpers ────────────────────────────────────────────────
@torch.no_grad()
def predict(model, ds):
    model.eval()
    probs = []
    for x, _ in DataLoader(ds, batch_size=64, shuffle=False):
        probs.append(torch.sigmoid(model(x.to(device))).squeeze(1).cpu().numpy())
    return np.concatenate(probs)

def pred_frame(ds, probs, set_name, split_name):
    df = ds.meta.copy()
    df.insert(0, "split", split_name)
    df.insert(1, "set", set_name)
    df["true_label"]   = df.pop("label").astype(int)     # 0 = HER2-, 1 = HER2+
    df["prob_HER2pos"] = probs
    df["pred_label"]   = (probs >= THRESHOLD).astype(int)
    return df[["split", "set", "spot_id", "patient", "section",
               "true_label", "pred_label", "prob_HER2pos"]]

# ── one split ─────────────────────────────────────────────────────────
def eval_loss(model, dl):
    """Mean BCE over a loader, no gradients, no augmentation."""
    model.eval()
    bce = nn.BCEWithLogitsLoss(reduction="sum")
    total, n = 0.0, 0
    with torch.no_grad():
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            total += bce(model(x).squeeze(1), y).item()
            n     += len(y)
    return total / n

def run_split(split_dir, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    out = out_root / split_dir.name; out.mkdir(exist_ok=True)

    tr_pats  = pd.read_csv(split_dir / "train_patients.csv")
    te_pats  = pd.read_csv(split_dir / "test_patients.csv")
    all_pats = pd.concat([tr_pats, te_pats])
    her2_map = dict(zip(all_pats["patient"].astype(int), all_pats["her2"]))

    train_spots = load_spots(split_dir / "train_spots_tumor.csv", her2_map)
    test_spots  = load_spots(split_dir / "test_spots_tumor.csv",  her2_map)

    mean, std = channel_stats(HER2Dataset(train_spots))
    tr_ds = HER2Dataset(train_spots, augment=True,  mean=mean, std=std)
    te_ds = HER2Dataset(test_spots,  augment=False, mean=mean, std=std)

    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
    te_dl = DataLoader(te_ds, batch_size=64,         shuffle=False, num_workers=NUM_WORKERS)

    model = HER2CNN().to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
    crit  = nn.BCEWithLogitsLoss()

    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        total = 0.0
        for x, y in tr_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(model(x).squeeze(1), y)
            loss.backward()
            opt.step()
            total += loss.item() * len(y)
        sched.step()
        train_loss = total / len(tr_ds)
        test_loss  = eval_loss(model, te_dl)     # diagnostic only, not used for selection

        history.append(dict(split=split_dir.name, epoch=epoch,
                            train_loss=round(train_loss, 5),
                            test_loss=round(test_loss, 5)))
        print(f"  {split_dir.name} epoch {epoch:02d}/{MAX_EPOCHS} | "
              f"train {train_loss:.4f} | test {test_loss:.4f}")

    torch.save(dict(state_dict=model.state_dict(), mean=mean, std=std),
               out / "model.pt")

    pred_te = pred_frame(te_ds, predict(model, te_ds), "test", split_dir.name)
    hist_df = pd.DataFrame(history)
    hist_df.to_csv(out / "epoch_loss.csv", index=False)
    pred_te.to_csv(out / "predictions_test.csv", index=False)
    print(f"  {split_dir.name} | train spots {len(tr_ds)} | test spots {len(te_ds)}")
    return hist_df, pred_te

all_hist, all_te = [], []
for i in range(15, 20):
    h, pte = run_split(splits_root / f"split_{i:02d}", seed=100 + i)
    all_hist.append(h); all_te.append(pte)
pd.concat(all_hist).to_csv(out_root / "epoch_loss_all.csv", index=False)
pd.concat(all_te).to_csv(out_root / "predictions_test_all.csv", index=False)







